In [1]:
# Importing necessary libraries
import pandas as pd
import sqlite3

conn = sqlite3.connect(":memory:")

def run(query): return pd.read_sql(query, conn)

In [2]:
# Loading the Excel file into pandas
df = pd.read_excel(r"C:\Projects\coffee-shop-analysis\data\Coffee Shop Sales.xlsx")
df.head(10)


,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
0,1,2023-01-01,07:06:11,2,5,Lower Manhattan,32,3.00,Coffee,Gourmet brewed coffee,Ethiopia Rg
1,2,2023-01-01,07:08:56,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2,3,2023-01-01,07:14:04,2,5,Lower Manhattan,59,4.50,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
3,4,2023-01-01,07:20:24,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm
4,5,2023-01-01,07:22:41,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
5,6,2023-01-01,07:22:41,1,5,Lower Manhattan,77,3.00,Bakery,Scone,Oatmeal Scone
6,7,2023-01-01,07:25:49,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm
7,8,2023-01-01,07:33:34,2,5,Lower Manhattan,28,2.00,Coffee,Gourmet brewed coffee,Columbian Medium Roast Sm
8,9,2023-01-01,07:39:13,1,5,Lower Manhattan,39,4.25,Coffee,Barista Espresso,Latte Rg
9,10,2023-01-01,07:39:34,2,5,Lower Manhattan,58,3.50,Drinking Chocolate,Hot chocolate,Dark chocolate Rg


In [3]:
# Creating a new column "revenue" in DataFrame
df['revenue'] = df['transaction_qty']*df['unit_price']
print(df[['transaction_qty','unit_price','revenue']].head())

   transaction_qty  unit_price  revenue
0                2         3.0      6.0
1                2         3.1      6.2
2                2         4.5      9.0
3                1         2.0      2.0
4                2         3.1      6.2


In [4]:
# Loading DataFrame in SQL
df.to_sql("coffee", conn , index=False, if_exists="replace")

run("SELECT * FROM coffee LIMIT 10")

,transaction_id,transaction_date,transaction_time,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,revenue
0,1,2023-01-01 00:00:00,07:06:11.000000,2,5,Lower Manhattan,32,3.00,Coffee,Gourmet brewed coffee,Ethiopia Rg,6.00
1,2,2023-01-01 00:00:00,07:08:56.000000,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.20
2,3,2023-01-01 00:00:00,07:14:04.000000,2,5,Lower Manhattan,59,4.50,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,9.00
3,4,2023-01-01 00:00:00,07:20:24.000000,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.00
4,5,2023-01-01 00:00:00,07:22:41.000000,2,5,Lower Manhattan,57,3.10,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,6.20
5,6,2023-01-01 00:00:00,07:22:41.000000,1,5,Lower Manhattan,77,3.00,Bakery,Scone,Oatmeal Scone,3.00
6,7,2023-01-01 00:00:00,07:25:49.000000,1,5,Lower Manhattan,22,2.00,Coffee,Drip coffee,Our Old Time Diner Blend Sm,2.00
7,8,2023-01-01 00:00:00,07:33:34.000000,2,5,Lower Manhattan,28,2.00,Coffee,Gourmet brewed coffee,Columbian Medium Roast Sm,4.00
8,9,2023-01-01 00:00:00,07:39:13.000000,1,5,Lower Manhattan,39,4.25,Coffee,Barista Espresso,Latte Rg,4.25
9,10,2023-01-01 00:00:00,07:39:34.000000,2,5,Lower Manhattan,58,3.50,Drinking Chocolate,Hot chocolate,Dark chocolate Rg,7.00


In [5]:
# Understanding the table scheme through PRAGMA
run("PRAGMA table_info(coffee)")

,cid,name,type,notnull,dflt_value,pk
0,0,transaction_id,INTEGER,0,None,0
1,1,transaction_date,TIMESTAMP,0,None,0
2,2,transaction_time,TIME,0,None,0
3,3,transaction_qty,INTEGER,0,None,0
4,4,store_id,INTEGER,0,None,0
5,5,store_location,TEXT,0,None,0
6,6,product_id,INTEGER,0,None,0
7,7,unit_price,REAL,0,None,0
8,8,product_category,TEXT,0,None,0
9,9,product_type,TEXT,0,None,0


In [6]:
# Find top 3 product types per store location by revenue using a window function

run("""SELECT * FROM 
(SELECT store_location,
product_type, SUM(revenue) AS revenue ,
RANK() OVER(PARTITION BY 
store_location ORDER BY SUM(revenue) DESC) 
AS Store_rank FROM coffee GROUP BY store_location, product_type) 
WHERE Store_rank <= 3""")

,store_location,product_type,revenue,Store_rank
0,Astoria,Barista Espresso,27935.00,1
1,Astoria,Brewed Chai tea,27427.90,2
2,Astoria,Hot chocolate,26335.25,3
3,Hell's Kitchen,Barista Espresso,32420.20,1
4,Hell's Kitchen,Brewed Chai tea,25645.30,2
5,Hell's Kitchen,Hot chocolate,23586.25,3
6,Lower Manhattan,Barista Espresso,31051.00,1
7,Lower Manhattan,Brewed Chai tea,24008.75,2
8,Lower Manhattan,Gourmet brewed coffee,23201.20,3


In [7]:
result = run("""SELECT * FROM 
(SELECT store_location,
product_type, SUM(revenue) AS revenue ,
RANK() OVER(PARTITION BY 
store_location ORDER BY SUM(revenue) DESC) 
AS Store_rank FROM coffee GROUP BY store_location, product_type) 
WHERE Store_rank <= 3""")

print(len(result))

9


In [8]:
# CTE 
# 1. Rewrite yesterday's top-3-per-store query using a CTE
# RUN("""SELECT *
# FROM (
#     SELECT
#         store_location,
#         product_type,
#         SUM(revenue) AS revenue,

#         RANK() OVER (
#             PARTITION BY store_location
#             ORDER BY SUM(revenue) DESC
#         ) AS store_rank

#     FROM coffee
#     GROUP BY store_location, product_type
# )
# WHERE store_rank <= 3;""")

run("""
WITH ranked AS (SELECT store_location, product_type, SUM(revenue) AS revenue, 
RANK() OVER(PARTITION BY store_location ORDER BY SUM(revenue) DESC) AS store_rank 
FROM coffee GROUP BY store_location, product_type)

SELECT * FROM ranked WHERE store_rank <=3
""")

,store_location,product_type,revenue,store_rank
0,Astoria,Barista Espresso,27935.00,1
1,Astoria,Brewed Chai tea,27427.90,2
2,Astoria,Hot chocolate,26335.25,3
3,Hell's Kitchen,Barista Espresso,32420.20,1
4,Hell's Kitchen,Brewed Chai tea,25645.30,2
5,Hell's Kitchen,Hot chocolate,23586.25,3
6,Lower Manhattan,Barista Espresso,31051.00,1
7,Lower Manhattan,Brewed Chai tea,24008.75,2
8,Lower Manhattan,Gourmet brewed coffee,23201.20,3


In [9]:

# 2. Stack two CTEs: first one calculates revenue per store,
#    second one calculates revenue per product_type,
#    final SELECT joins both results side by side

run("""

WITH 
store_revenue AS (SELECT store_location, SUM(revenue) AS 
Total_store_revenue FROM coffee GROUP BY store_location)
,
product_revenue AS (Select product_type, SUM(revenue) AS 
Total_product_revenue FROM coffee GROUP BY product_type)

SELECT * FROM store_revenue CROSS JOIN product_revenue """)

,store_location,Total_store_revenue,product_type,Total_product_revenue
0,Astoria,232243.91,Barista Espresso,91406.20
1,Astoria,232243.91,Biscotti,19793.53
2,Astoria,232243.91,Black tea,2711.85
3,Astoria,232243.91,Brewed Black tea,47932.00
4,Astoria,232243.91,Brewed Chai tea,77081.95
...,...,...,...,...
82,Lower Manhattan,230057.25,Premium Beans,14583.50
83,Lower Manhattan,230057.25,Premium brewed coffee,38781.15
84,Lower Manhattan,230057.25,Regular syrup,6084.80
85,Lower Manhattan,230057.25,Scone,36866.12


In [10]:
# 3. Verify both produce correct output
run("SELECT store_location, SUM(revenue) AS Total_store_revenue FROM coffee GROUP BY store_location")

,store_location,Total_store_revenue
0,Astoria,232243.91
1,Hell's Kitchen,236511.17
2,Lower Manhattan,230057.25


In [11]:
run("Select product_type, SUM(revenue) AS Total_product_revenue FROM coffee GROUP BY product_type")

,product_type,Total_product_revenue
0,Barista Espresso,91406.20
1,Biscotti,19793.53
2,Black tea,2711.85
3,Brewed Black tea,47932.00
4,Brewed Chai tea,77081.95
5,Brewed Green tea,23852.50
6,Brewed herbal tea,47539.50
7,Chai tea,4301.25
8,Clothing,6163.00
9,Drinking Chocolate,2728.04


In [15]:
# 1. Add a revenue_tier column to each transaction:
#    - revenue >= 10 → 'High'
#    - revenue >= 5  → 'Medium'
#    - else          → 'Low'
#    Show 15 rows with store_location, product_type, revenue, revenue_tier

run("""
SELECT 
store_location, product_type, revenue, 
CASE 
    WHEN revenue >=10 THEN 'High' 
    WHEN revenue >= 5 THEN 'Medium' 
    ELSE 'Low' 
END AS revenue_tier

FROM coffee LIMIT 15
""")

,store_location,product_type,revenue,revenue_tier
0,Lower Manhattan,Gourmet brewed coffee,6.00,Medium
1,Lower Manhattan,Brewed Chai tea,6.20,Medium
2,Lower Manhattan,Hot chocolate,9.00,Medium
3,Lower Manhattan,Drip coffee,2.00,Low
4,Lower Manhattan,Brewed Chai tea,6.20,Medium
5,Lower Manhattan,Scone,3.00,Low
6,Lower Manhattan,Drip coffee,2.00,Low
7,Lower Manhattan,Gourmet brewed coffee,4.00,Low
8,Lower Manhattan,Barista Espresso,4.25,Low
9,Lower Manhattan,Hot chocolate,7.00,Medium


In [19]:
# 2. Combine CASE WHEN with GROUP BY:
#    Count how many transactions fall into each tier per store
#    (columns: store_location, tier, transaction_count)

run("""
WITH tiered_sales AS (SELECT 
store_location, product_type, revenue, 
CASE 
    WHEN revenue >=10 THEN 'High' 
    WHEN revenue >= 5 THEN 'Medium' 
    ELSE 'Low' 
END AS revenue_tier 
FROM coffee )

SELECT store_location, revenue_tier, COUNT(*) AS transaction_count 
FROM tiered_sales 
GROUP BY store_location, revenue_tier

""")

,store_location,revenue_tier,transaction_count
0,Astoria,High,865
1,Astoria,Low,32102
2,Astoria,Medium,17632
3,Hell's Kitchen,High,923
4,Hell's Kitchen,Low,32022
5,Hell's Kitchen,Medium,17790
6,Lower Manhattan,High,1817
7,Lower Manhattan,Low,29339
8,Lower Manhattan,Medium,16626
